# Stage 1b: marker-length datasets
Builds 5 new versions of the **same** Stage 1 dataset, one per marker length, into
`MyDrive/fyp_dna/data/stage1_markers`. Your Stage 1 data is not touched.

| Arm | Marker | Each 32-base block | Data bases | Marker share |
|---|---|---|---|---|
| marker_len04 | CAGT | 28 data + 4 | 112 | 12.5% |
| marker_len05 | TCATG | 27 data + 5 | 108 | 15.6% |
| marker_len06 | CTGTCA | 26 data + 6 | 104 | 18.8% |
| marker_len07 | CTGATCA | 25 data + 7 | 100 | 21.9% |
| marker_len12 | ACGTCAGTGCAT | 20 data + 12 | 80 | 37.5% |

Length 8 (`ACGTCAGT`) is `marker_periodic` in the Stage 1 data. Same payloads, same seeds, same error
positions as Stage 1. **No GPU needed.** Runtime > **Run all** (about 10 minutes). Paste the output of the last cell.

### Cell 1: mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
PROJECT = '/content/drive/MyDrive/fyp_dna'
CODE    = f'{PROJECT}/code'
REPO    = '/content/Deep-DNA-based-storage'
REF     = f'{PROJECT}/data/stage1_pilot'
OUT     = f'{PROJECT}/data/stage1_markers'
os.makedirs(CODE, exist_ok=True); os.makedirs(f'{PROJECT}/logs', exist_ok=True)
print('Stage 1 reference data found:', os.path.isfile(f'{REF}/marker_free/train/metadata.json'))
print('Output folder already exists:', os.path.isdir(OUT))

### Cell 2: write the dataset generator (now supports marker lengths)

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/stage1_generate.py
"""
Stage 1: paired dataset generator for the marker vs. marker-free DNAFormer study.

Makes three paired "arms" from the SAME random payloads and the SAME noise seeds:
  marker_free      128 payload bases, no markers
  marker_periodic  (24 payload + 8 marker) x 4   = 96 payload + 32 marker
  marker_endblock  96 payload, then the 4 markers together at the end (control)

Noisy reads come from DNAFormer's own simulator
(Deep-DNA-based-storage/DataGenerator/cluster_generator.py). We only give it our
own error rates; we do not edit the repository.

Usage (Colab or local):
  python stage1_generate.py --repo /content/Deep-DNA-based-storage --out /content/drive/MyDrive/fyp_dna/data/stage1_pilot
  python stage1_generate.py --repo ... --out ... --tiny      # quick smoke test
"""
import argparse
import csv
import hashlib
import json
import os
import random
import subprocess
import sys
import time

import numpy as np

# ---------------------------------------------------------------------------
# Fixed experiment settings (saved to config.json)
# ---------------------------------------------------------------------------
BASES = "ACGT"
STRAND_LEN = 128
MARKER = "ACGTCAGT"
MARKER_INTERVAL = 24          # payload bases between markers
N_MARKERS = 4
N_READS = 10
DELETION_LEVELS = [0.01, 0.03, 0.05, 0.08]
INSERTION_RATE = 0.01
SUBSTITUTION_RATE = 0.01
LONG_DELETION_RATE = 0.0      # all deletions are single-base; change later if wanted

ARMS = ["marker_free", "marker_periodic", "marker_endblock"]   # Stage 1 default

# Every arm: how the 128-base strand is built. "periodic" = 4 blocks of 32 bases,
# each block = (32 - L) data bases followed by the L-base marker. Payloads are always
# the FIRST data bases of the same 128-base random payload, and the noise seed is
# shared, so all arms are paired (same data, same error positions).
BLOCK = STRAND_LEN // N_MARKERS   # 32
ARM_SPECS = {
    "marker_free":     {"layout": "none",      "marker": ""},
    "marker_periodic": {"layout": "periodic",  "marker": "ACGTCAGT"},      # length 8 (Stage 4)
    "marker_endblock": {"layout": "end_block", "marker": "ACGTCAGT"},      # control
    "marker_len04":    {"layout": "periodic",  "marker": "CAGT"},
    "marker_len05":    {"layout": "periodic",  "marker": "TCATG"},
    "marker_len06":    {"layout": "periodic",  "marker": "CTGTCA"},
    "marker_len07":    {"layout": "periodic",  "marker": "CTGATCA"},
    "marker_len12":    {"layout": "periodic",  "marker": "ACGTCAGTGCAT"},
}
LENGTH_ARMS = ["marker_len04", "marker_len05", "marker_len06", "marker_len07", "marker_len12"]
ALL_ARMS = list(ARM_SPECS)

# Seed ranges never overlap between splits, so the same payload cannot appear in two splits.
SPLIT_SEED_BASE = {"train": 10_000_000, "val": 20_000_000, "test": 30_000_000}
LEVEL_SEED_STRIDE = 1_000_000  # each deletion level gets its own block of seeds
NOISE_SEED_OFFSET = 1_000_000_000
SHUFFLE_SEED = 12345

PILOT_SIZES = {"train": 2500, "val": 250, "test": 1000}   # clusters PER deletion level
TINY_SIZES = {"train": 20, "val": 5, "test": 10}


def level_tag(d):
    return "del%02d" % round(d * 100)


# ---------------------------------------------------------------------------
# Strand construction
# ---------------------------------------------------------------------------
def make_payload128(payload_seed):
    rng = np.random.default_rng(payload_seed)
    return "".join(BASES[i] for i in rng.integers(0, 4, STRAND_LEN))


def arm_marker(arm):
    return ARM_SPECS[arm]["marker"]


def data_per_block(arm):
    return BLOCK - len(arm_marker(arm))


def build_strand(arm, payload128):
    """Return (clean_strand, payload, marker_positions)."""
    spec = ARM_SPECS[arm]
    m = spec["marker"]
    if spec["layout"] == "none":
        return payload128, payload128, []
    n_data = STRAND_LEN - N_MARKERS * len(m)
    payload = payload128[:n_data]               # same seed -> first bases shared by all arms
    if spec["layout"] == "periodic":
        d = data_per_block(arm)
        parts, positions = [], []
        for k in range(N_MARKERS):
            parts.append(payload[k * d:(k + 1) * d])
            start = sum(len(p) for p in parts)
            parts.append(m)
            positions.append([start, start + len(m)])
        return "".join(parts), payload, positions
    if spec["layout"] == "end_block":
        start = len(payload)
        positions = [[start + k * len(m), start + (k + 1) * len(m)] for k in range(N_MARKERS)]
        return payload + m * N_MARKERS, payload, positions
    raise ValueError(arm)


def remove_markers(arm, strand):
    """Payload-only view of a 128-base strand (used by later stages for scoring)."""
    spec = ARM_SPECS[arm]
    if spec["layout"] == "none":
        return strand
    n_data = STRAND_LEN - N_MARKERS * len(spec["marker"])
    if spec["layout"] == "periodic":
        d = data_per_block(arm)
        return "".join(strand[k * BLOCK:k * BLOCK + d] for k in range(N_MARKERS))
    if spec["layout"] == "end_block":
        return strand[:n_data]
    raise ValueError(arm)


def payload_to_bytes(payload):
    """2 bits per base (A=00 C=01 G=10 T=11). 128 bases -> 32 bytes, 96 bases -> 24 bytes."""
    bits = "".join(format(BASES.index(b), "02b") for b in payload)
    return bytes(int(bits[i:i + 8], 2) for i in range(0, len(bits), 8))


# ---------------------------------------------------------------------------
# DNAFormer simulator wrapper
# ---------------------------------------------------------------------------
def load_simulator(repo):
    gen_dir = os.path.join(repo, "DataGenerator")
    if not os.path.isfile(os.path.join(gen_dir, "cluster_generator.py")):
        sys.exit("ERROR: cluster_generator.py not found in %s. Check --repo." % gen_dir)
    sys.path.insert(0, gen_dir)
    from cluster_generator import ClusterGenerator  # noqa: E402
    return ClusterGenerator


def error_rates(deletion_rate):
    """Rates in the simulator's own format.

    The simulator gives each base one error chance equal to the SUM of the general
    rates, then picks the error type with the per-base weights. Using the same numbers
    for both makes the per-base probabilities exactly sub/ins/del.
    """
    general = {"s": SUBSTITUTION_RATE, "i": INSERTION_RATE, "d": deletion_rate, "ld": LONG_DELETION_RATE}
    per_base = {b: {"s": SUBSTITUTION_RATE, "i": 0.25, "pi": INSERTION_RATE,
                    "d": deletion_rate, "ld": LONG_DELETION_RATE} for b in BASES}
    # 'i' only chooses WHICH base is inserted (uniform here); 'pi' is the insertion rate.
    return general, per_base


def simulate_reads(ClusterGenerator, clean, deletion_rate, noise_seed):
    random.seed(noise_seed)          # the simulator uses Python's global random module
    np.random.seed(noise_seed % (2 ** 32))
    general, per_base = error_rates(deletion_rate)
    gen = ClusterGenerator(general, per_base, clean, min_copies=N_READS, max_copies=N_READS)
    gen.cluster_size = N_READS       # the repo's randint(min, max+1) could give 11; force exactly 10
    gen.generate_cluster(delta=0)    # delta=0: no random jitter of the rates
    assert len(gen.copies) == N_READS
    return list(gen.copies)


# ---------------------------------------------------------------------------
# Cluster plan
# ---------------------------------------------------------------------------
def plan_clusters(sizes):
    """List of dicts: split, subset, deletion_rate, payload_seed, noise_seed, pair_id."""
    plan = []
    for split in ["train", "val", "test"]:
        for li, d in enumerate(DELETION_LEVELS):
            for i in range(sizes[split]):
                seed = SPLIT_SEED_BASE[split] + li * LEVEL_SEED_STRIDE + i
                subset = split if split != "test" else "test_" + level_tag(d)
                plan.append({
                    "pair_id": "%s_%s_%06d" % (split, level_tag(d), i),
                    "split": split,
                    "subset": subset,
                    "deletion_rate": d,
                    "payload_seed": seed,
                    "noise_seed": seed + NOISE_SEED_OFFSET,
                })
    # Mix the deletion levels inside train and val (fixed shuffle seed).
    rng = random.Random(SHUFFLE_SEED)
    for split in ["train", "val"]:
        idx = [k for k, c in enumerate(plan) if c["split"] == split]
        items = [plan[k] for k in idx]
        rng.shuffle(items)
        for k, c in zip(idx, items):
            plan[k] = c
    return plan


def make_cluster(ClusterGenerator, arm, c):
    payload128 = make_payload128(c["payload_seed"])
    clean, payload, marker_pos = build_strand(arm, payload128)
    reads = simulate_reads(ClusterGenerator, clean, c["deletion_rate"], c["noise_seed"])
    n_marker = len(clean) - len(payload)
    return {
        "cluster_id": "%s__%s" % (arm, c["pair_id"]),
        "pair_id": c["pair_id"],
        "arm": arm,
        "split": c["split"],
        "subset": c["subset"],
        "seed": c["payload_seed"],
        "payload_seed": c["payload_seed"],
        "noise_seed": c["noise_seed"],
        "marker_present": ARM_SPECS[arm]["layout"] != "none",
        "marker_layout": ARM_SPECS[arm]["layout"],
        "marker_sequence": arm_marker(arm),
        "marker_interval": data_per_block(arm) if ARM_SPECS[arm]["layout"] == "periodic" else None,
        "marker_positions": marker_pos,
        "marker_overhead": n_marker / len(clean),
        "payload_length": len(payload),
        "strand_length": len(clean),
        "clean_strand": clean,
        "payload_strand": payload,
        "payload_bytes_hex": payload_to_bytes(payload).hex(),
        "noisy_reads": reads,
        "read_lengths": [len(r) for r in reads],
        "deletion_rate": c["deletion_rate"],
        "insertion_rate": INSERTION_RATE,
        "substitution_rate": SUBSTITUTION_RATE,
        "long_deletion_rate": LONG_DELETION_RATE,
    }


# ---------------------------------------------------------------------------
# Writing
# ---------------------------------------------------------------------------
def write_subset(folder, clusters):
    os.makedirs(folder, exist_ok=True)
    with open(os.path.join(folder, "clean.fasta"), "w", newline="\n") as f:
        for c in clusters:
            f.write(">%s\n%s\n" % (c["cluster_id"], c["clean_strand"]))
    with open(os.path.join(folder, "noisy_reads.fasta"), "w", newline="\n") as f:
        for c in clusters:
            for j, r in enumerate(c["noisy_reads"]):
                f.write(">%s|read_%d\n%s\n" % (c["cluster_id"], j, r))
    with open(os.path.join(folder, "metadata.json"), "w", newline="\n") as f:
        json.dump(clusters, f, separators=(",", ":"))


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def repo_commit(repo):
    try:
        return subprocess.check_output(["git", "-C", repo, "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True, help="path to Deep-DNA-based-storage")
    ap.add_argument("--out", required=True, help="output folder")
    ap.add_argument("--tiny", action="store_true", help="very small smoke-test dataset")
    ap.add_argument("--arms", default=",".join(ARMS),
                    help="comma-separated arms to build (default: the 3 Stage 1 arms); "
                         "'lengths' = the marker-length arms %s" % LENGTH_ARMS)
    args = ap.parse_args()
    arms = LENGTH_ARMS if args.arms == "lengths" else [a.strip() for a in args.arms.split(",")]
    for a in arms:
        if a not in ARM_SPECS:
            sys.exit("ERROR: unknown arm %s. Known: %s" % (a, ALL_ARMS))

    sizes = TINY_SIZES if args.tiny else PILOT_SIZES
    ClusterGenerator = load_simulator(args.repo)
    plan = plan_clusters(sizes)
    os.makedirs(args.out, exist_ok=True)

    t0 = time.time()
    manifest_rows = []
    for arm in arms:
        by_subset = {}
        for k, c in enumerate(plan):
            cl = make_cluster(ClusterGenerator, arm, c)
            by_subset.setdefault(c["subset"], []).append(cl)
            if (k + 1) % 2000 == 0:
                print("  %s: %d/%d clusters (%.0fs)" % (arm, k + 1, len(plan), time.time() - t0), flush=True)
        for subset, clusters in by_subset.items():
            folder = os.path.join(args.out, arm, subset)
            write_subset(folder, clusters)
            for cl in clusters:
                manifest_rows.append({
                    "cluster_id": cl["cluster_id"], "pair_id": cl["pair_id"], "arm": arm,
                    "split": cl["split"], "subset": subset,
                    "deletion_rate": cl["deletion_rate"], "insertion_rate": cl["insertion_rate"],
                    "substitution_rate": cl["substitution_rate"],
                    "payload_seed": cl["payload_seed"], "noise_seed": cl["noise_seed"],
                    "marker_present": cl["marker_present"], "marker_layout": cl["marker_layout"],
                    "marker_sequence": cl["marker_sequence"], "marker_interval": cl["marker_interval"],
                    "payload_length": cl["payload_length"], "strand_length": cl["strand_length"],
                    "marker_overhead": cl["marker_overhead"], "n_reads": len(cl["noisy_reads"]),
                    "mean_read_length": round(float(np.mean(cl["read_lengths"])), 2),
                    "folder": os.path.join(arm, subset).replace("\\", "/"),
                })
        print("%s done (%.0fs)" % (arm, time.time() - t0), flush=True)

    with open(os.path.join(args.out, "dataset_manifest.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(manifest_rows[0].keys()))
        w.writeheader()
        w.writerows(manifest_rows)

    config = {
        "stage": 1, "mode": "tiny" if args.tiny else "pilot",
        "clusters_per_level": sizes, "deletion_levels": DELETION_LEVELS,
        "insertion_rate": INSERTION_RATE, "substitution_rate": SUBSTITUTION_RATE,
        "long_deletion_rate": LONG_DELETION_RATE, "strand_length": STRAND_LEN,
        "marker": MARKER, "marker_interval": MARKER_INTERVAL, "n_markers": N_MARKERS,
        "n_reads": N_READS, "arms": arms,
        "arm_specs": {a: dict(ARM_SPECS[a], data_bases=len(build_strand(a, "A" * STRAND_LEN)[1]))
                      for a in arms},
        "split_seed_base": SPLIT_SEED_BASE,
        "level_seed_stride": LEVEL_SEED_STRIDE, "noise_seed_offset": NOISE_SEED_OFFSET,
        "shuffle_seed": SHUFFLE_SEED, "simulator": "DataGenerator/cluster_generator.py",
        "simulator_repo_commit": repo_commit(args.repo),
        "python": sys.version.split()[0], "numpy": np.__version__,
    }
    with open(os.path.join(args.out, "config.json"), "w", newline="\n") as f:
        json.dump(config, f, indent=2)

    checksums = {}
    for root, _, files in os.walk(args.out):
        for name in files:
            if name in ("checksums.json",):
                continue
            p = os.path.join(root, name)
            checksums[os.path.relpath(p, args.out).replace("\\", "/")] = sha256(p)
    with open(os.path.join(args.out, "checksums.json"), "w", newline="\n") as f:
        json.dump(dict(sorted(checksums.items())), f, indent=1)

    print("\nDONE: %d clusters per arm x %d arms in %.0fs -> %s"
          % (len(plan), len(arms), time.time() - t0, args.out))


if __name__ == "__main__":
    main()

### Cell 3: write the Stage 1b validation script

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/stage1b_validate.py
"""
Stage 1b validation: checks the marker-length datasets against the original Stage 1 data.

Each new arm must use the SAME payloads (first data bases of the marker_free payload),
the SAME seeds and error rates, and get errors at the SAME positions (so every read
has exactly the same length as the matching marker_free read).

Usage:
  python stage1b_validate.py --repo REPO --data NEW_DATA --ref STAGE1_DATA
"""
import argparse
import collections
import hashlib
import json
import os
import random
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import stage1_generate as G  # noqa: E402

FAILS = []


def check(ok, msg, quiet=False):
    if not ok or not quiet:
        print(("  [PASS] " if ok else "  [FAIL] ") + msg)
    if not ok:
        FAILS.append(msg)


def read_fasta(path):
    out, name = {}, None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                name = line[1:]
            elif line:
                out[name] = line
    return out


def load(data, arm, sub):
    with open(os.path.join(data, arm, sub, "metadata.json")) as f:
        return json.load(f)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--data", required=True, help="new marker-length dataset folder")
    ap.add_argument("--ref", required=True, help="original Stage 1 dataset folder (has marker_free)")
    args = ap.parse_args()

    cfg = json.load(open(os.path.join(args.data, "config.json")))
    arms, sizes, levels = cfg["arms"], cfg["clusters_per_level"], cfg["deletion_levels"]
    subsets = ["train", "val"] + ["test_" + G.level_tag(d) for d in levels]
    print("New data: %s (%s)\nReference: %s\nArms: %s" % (args.data, cfg["mode"], args.ref, arms))

    print("\n1. Reference marker_free data")
    ref = {}
    for sub in subsets:
        ref.update({c["pair_id"]: c for c in load(args.ref, "marker_free", sub)})
    check(len(ref) == sum(sizes[s if not s.startswith("test") else "test"] * (1 if s.startswith("test") else len(levels))
                          for s in subsets), "loaded %d reference clusters" % len(ref))

    rows = []
    for arm in arms:
        m = G.arm_marker(arm)
        L, d = len(m), G.data_per_block(arm)
        n_data = 128 - 4 * L
        want_pos = [[k * 32 + d, k * 32 + 32] for k in range(4)]
        print("\n2. %s  marker %s (length %d): 4 x (%d data + %d marker) = %d data bases"
              % (arm, m, L, d, L, n_data))
        total, chance = 0, 0
        ok = collections.Counter()
        for sub in subsets:
            folder = os.path.join(args.data, arm, sub)
            meta = load(args.data, arm, sub)
            want = sizes["test"] if sub.startswith("test") else sizes[sub] * len(levels)
            check(len(meta) == want, "%s/%s: %d clusters (expected %d)" % (arm, sub, len(meta), want), quiet=True)
            clean_fa = read_fasta(os.path.join(folder, "clean.fasta"))
            reads_fa = read_fasta(os.path.join(folder, "noisy_reads.fasta"))
            check(len(clean_fa) == len(meta) and len(reads_fa) == 10 * len(meta)
                  and all(clean_fa[c["cluster_id"]] == c["clean_strand"] for c in meta)
                  and all(reads_fa["%s|read_%d" % (c["cluster_id"], j)] == r
                          for c in meta for j, r in enumerate(c["noisy_reads"])),
                  "%s/%s: FASTA files match metadata.json" % (arm, sub), quiet=True)
            for c in meta:
                r = ref[c["pair_id"]]
                s = c["clean_strand"]
                ok["len128"] += len(s) == 128 and set(s) <= set("ACGT")
                ok["markers"] += all(s[a:b] == m for a, b in want_pos) and c["marker_positions"] == want_pos
                ok["payload"] += (c["payload_strand"] == r["payload_strand"][:n_data]
                                  and G.remove_markers(arm, s) == c["payload_strand"]
                                  and len(c["payload_strand"]) == n_data)
                ok["same_seeds_rates"] += (c["payload_seed"] == r["payload_seed"] and c["noise_seed"] == r["noise_seed"]
                                           and c["deletion_rate"] == r["deletion_rate"] and c["subset"] == r["subset"]
                                           and c["insertion_rate"] == r["insertion_rate"]
                                           and c["substitution_rate"] == r["substitution_rate"])
                ok["same_error_positions"] += [len(x) for x in c["noisy_reads"]] == [len(x) for x in r["noisy_reads"]]
                ok["ten_reads"] += len(c["noisy_reads"]) == 10
                chance += m in c["payload_strand"]
                total += 1
        check(all(v == total for v in ok.values()) and len(ok) == 6,
              "all %d clusters: 128 bases, markers at %s, payload = first %d bases of marker_free payload, "
              "same seeds/rates, 10 reads" % (total, [p[0] for p in want_pos], n_data))
        check(ok["same_error_positions"] == total,
              "every read has the same length as the matching marker_free read (same error positions)")
        rows.append((arm, m, L, n_data, 4 * L / 128, chance / total))

    print("\n3. Reproducibility")
    ClusterGenerator = G.load_simulator(args.repo)
    plan = {c["pair_id"]: c for c in G.plan_clusters(sizes)}
    rng = random.Random(0)
    regen = True
    for arm in arms:
        meta = load(args.data, arm, "test_del08") + load(args.data, arm, "train")[:200]
        for c in rng.sample(meta, 10):
            regen &= G.make_cluster(ClusterGenerator, arm, plan[c["pair_id"]]) == c
    check(regen, "regenerating 10 random clusters per arm from their seeds gives identical clusters")
    sums = json.load(open(os.path.join(args.data, "checksums.json")))
    bad = [p for p, h in sums.items()
           if hashlib.sha256(open(os.path.join(args.data, p), "rb").read()).hexdigest() != h]
    check(not bad, "all %d files match checksums.json" % len(sums))

    print("\n4. Summary")
    print("   arm            marker        length  data bases  marker share  payloads containing marker by chance")
    print("   %-14s %-13s %6d  %10d  %11.1f%%  %s" % ("marker_free", "-", 0, 128, 0.0, "-"))
    for arm, m, L, n, ov, ch in sorted(rows, key=lambda r: r[2]):
        print("   %-14s %-13s %6d  %10d  %11.1f%%  %.2f%%" % (arm, m, L, n, 100 * ov, 100 * ch))
    print("   (length 8, ACGTCAGT: marker_periodic in the Stage 1 data, 96 data bases, 25.0%)")

    print("\n" + ("ALL STAGE 1b CHECKS PASSED" if not FAILS else
                  "%d CHECK(S) FAILED:\n  - " % len(FAILS) + "\n  - ".join(FAILS)))
    sys.exit(1 if FAILS else 0)


if __name__ == "__main__":
    main()

### Cell 4: get DNAFormer repo (same pinned commit as Stage 1)

In [ ]:
!rm -rf {REPO}
!git clone -q https://github.com/itaiorr/Deep-DNA-based-storage {REPO}
!git -C {REPO} checkout -q 167a68261719ea458e79c40fcc485ae768c1a64a
!git -C {REPO} rev-parse HEAD

### Cell 5: build the 5 marker-length datasets in Drive (about 8 minutes, 301 MB)

In [ ]:
!python {CODE}/stage1_generate.py --repo {REPO} --out {OUT} --arms lengths 2>&1 | tail -3

### Cell 6: validate against Stage 1. Paste the whole output of this cell back to Claude.

In [ ]:
!python {CODE}/stage1b_validate.py --repo {REPO} --data {OUT} --ref {REF} 2>&1 | tee {PROJECT}/logs/stage1b_validation.txt
!du -sh {OUT}